In [19]:
import pandas as pd
import numpy as np

In [20]:
# Simulate realistic stock data with issues
np.random.seed(42)

# AAPL: full data, 252 trading days
dates_aapl = pd.bdate_range('2024-01-01', periods=252)
prices_aapl = 150 * (1 + np.random.randn(252).cumsum() * 0.01)
aapl = pd.Series(prices_aapl, index=dates_aapl, name='AAPL')

# TSLA: started tracking late, only 200 days
dates_tsla = pd.bdate_range('2024-03-15', periods=200)
prices_tsla = 2800 * (1 + np.random.randn(200).cumsum() * 0.015)
tsla = pd.Series(prices_tsla, index=dates_tsla, name='TSLA')

# MSFT: has 10 random missing days (trading halts, data gaps)
dates_msft = pd.bdate_range('2024-01-01', periods=252)
missing_indices = np.random.choice(252, size=10, replace=False)
dates_msft_filtered = dates_msft.delete(missing_indices)
prices_msft = 180 * (1 + np.random.randn(len(dates_msft_filtered)).cumsum() * 0.008)
msft = pd.Series(prices_msft, index=dates_msft_filtered, name='MSFT')

In [21]:
# End of Unit Deliverable

#function
def compute_log_returns(prices_series):
    #computing log returns - since we're taking log we can actually just sum them
    log_prices = np.log(prices_series)
    log_returns = log_prices.diff()
    return log_returns


def compute_multi_stock_log_returns(price_dict, min_overlap=20):
    #looping through the dictionary
    for ticker, prices in price_dict.items():
        #Key, Value
        #checking if it's a series
        if not isinstance(prices, pd.Series):
            raise ValueError(f"{ticker} is not a pd.Series")
        #checking if index is DateTimeIndex
        if not isinstance(prices.index, pd.DatetimeIndex):
            raise ValueError(f"{ticker} does not have DatetimeIndex")
        #checking if at least 2 data points
        if len(prices) < 2:
            raise ValueError(f"{ticker} has fewer than 2 data points")

        

    # Apply compute_log_returns to each stock
    log_returns_dict = {
        ticker: compute_log_returns(prices)
        for ticker, prices in price_dict.items()
    }

    returns_df = pd.DataFrame(log_returns_dict)

    # How many unique dates across ALL stocks?
    total_dates = len(returns_df)  # DataFrame has all dates (union)

    # Dates where NO stock has NaN
    common_dates = len(returns_df.dropna())

    # Or count rows with no NaNs:
    common_dates = returns_df.notna().all(axis=1).sum()

    # For each stock, what % of total dates do they cover?
    per_stock_coverage = {}

    for ticker in returns_df.columns:
        valid_count = returns_df[ticker].count()  # Non-NaN values
        coverage_pct = (valid_count / total_dates) * 100
        per_stock_coverage[ticker] = coverage_pct

    # Result: {'AAPL': 100.0, 'TSLA': 79.4, 'MSFT': 96.0}

    # For each pair of columns, count rows where both are not NaN
    pairwise_overlap = pd.DataFrame(
        index=returns_df.columns,
        columns=returns_df.columns,
        dtype=int
    )

    for col1 in returns_df.columns:
        for col2 in returns_df.columns:
            # How many rows where both col1 and col2 are not NaN?
            overlap = returns_df[[col1, col2]].notna().all(axis=1).sum()
            pairwise_overlap.loc[col1, col2] = overlap

    alignment_report = {
        'total_dates': total_dates,
        'common_dates': common_dates,
        'per_stock_coverage': per_stock_coverage,
        'pairwise_overlap': pairwise_overlap
    }

    return returns_df, alignment_report

# Result: a dictionary where each value is a Series of log returns
# {'AAPL': aapl_log_returns, 'TSLA': tsla_log_returns, 'MSFT': msft_log_returns}


#what get's passed into the function
    #dictionary Key: Values
price_dict = {
    'AAPL': aapl,    # pd.Series with DatetimeIndex
    'TSLA': tsla,    # pd.Series with DatetimeIndex  
    'MSFT': msft     # pd.Series with DatetimeIndex
}

In [22]:
# Should print without errors
returns, report = compute_multi_stock_log_returns(price_dict)

# Check returns
print(f"Returns type: {type(returns)}")  # Should be DataFrame
print(f"Returns shape: {returns.shape}")  # Should be (252, 3) or similar

# Check report
print(f"Report keys: {report.keys()}")  # Should have 4 keys
print(f"Total dates: {report['total_dates']}")

Returns type: <class 'pandas.DataFrame'>
Returns shape: (254, 3)
Report keys: dict_keys(['total_dates', 'common_dates', 'per_stock_coverage', 'pairwise_overlap'])
Total dates: 254
